In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
print('Ollama installed')

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,765 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,029 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-driv

In [ ]:
import subprocess, time, requests

LLAVA_MODEL = 'qwen3-vl:8b'

server = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print('Waiting for Ollama server...', end='')
for _ in range(20):
    try:
        requests.get('http://localhost:11434', timeout=1)
        print('ready!')
        break
    except Exception:
        print('.', end='', flush=True)
        time.sleep(1)
else:
    raise RuntimeError('Ollama server did not start in time')

# Pull LLaVA model (downloads ~4-5 GB, takes 1-3 min)
print(f'Pulling {LLAVA_MODEL} — this may take a few minutes...')
!ollama pull {LLAVA_MODEL}
print(f'{LLAVA_MODEL} ready')

Waiting for Ollama server........ready!
Pulling qwen3-vl:8b — this may take a few minutes...

qwen3-vl:8b ready


In [ ]:
!pip install pyngrok -q

from pyngrok import ngrok, conf

NGROK_AUTH_TOKEN = '***'

conf.get_default().auth_token = NGROK_AUTH_TOKEN
ngrok.kill()
tunnel = ngrok.connect("127.0.0.1:11434", host_header="localhost:11434")

OLLAMA_BASE_URL = tunnel.public_url
print('Ollama is publicly available at:')
print(f'{OLLAMA_BASE_URL}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted at /content/drive')

Mounted at /content/drive
Google Drive mounted at /content/drive


In [ ]:
import os

EXDARK_IMAGE_DIR = '/content/drive/MyDrive/ExDark/ExDark/'
OUTPUT_DIR       = '/content/drive/MyDrive/ExDark/ExDarkAnnotation-qwen3vl'
FAILED_LOG_PATH  = '/content/drive/MyDrive/ExDark/failed_images.txt'

MODEL_NAME = LLAVA_MODEL
OLLAMA_API_URL = f"{tunnel.public_url}/api/chat"

SYSTEM_PROMPT = """You are a Computer Vision expert. Look at this image and describe the main object or scene in ONE short English sentence.

RULES:
- Output exactly one sentence.
- No quotes, no JSON, no markdown, no labels like "Description:".
- No introduction, no explanation, just the sentence itself."""

MAX_RETRIES = 5
BASE_DELAY  = 2.0

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Output dir:', OUTPUT_DIR)
print('Image dir :', EXDARK_IMAGE_DIR)
print('Using Model:', MODEL_NAME)
print('API Endpoint:', OLLAMA_API_URL)

Output dir: /content/drive/MyDrive/ExDark/ExDarkAnnotation-qwen3vl
Image dir : /content/drive/MyDrive/ExDark/ExDark/
Using Model: qwen3-vl:8b
API Endpoint: https://going-seventeen-deceit.ngrok-free.dev/api/chat


In [ ]:
import re, json, base64, time, requests

def log_failure(image_path: str, error_msg: str):
    """Append a failed image path + error to the failure log."""
    with open(FAILED_LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(f"{image_path} | Error: {error_msg}\n")

def clean_to_sentence(raw_text: str) -> str:
    """Normalize raw model output to a single clean sentence."""
    text = raw_text.strip()

    if text.startswith('```'):
        text = text.split('\n', 1)[-1]
        if text.rstrip().endswith('```'):
            text = text.rstrip()[:-3]
        text = text.strip()

    quote_chars = '"\'“”‘’'
    if len(text) >= 2 and text[0] in quote_chars and text[-1] in quote_chars:
        text = text[1:-1].strip()

    text = text.splitlines()[0].strip()

    match = re.search(r'^[^.!?]*[.!?]', text)
    if match:
        text = match.group(0)

    text = text.strip().strip(quote_chars).strip()

    if not text:
        raise ValueError('Model returned empty output after cleaning.')

    return text

def wrap_as_annotation(sentence: str) -> dict:
    """Wrap sentence in control tokens and return annotation dict."""
    return {'description': f'<|desc_start|> {sentence} <|desc_end|>'}

def encode_image_b64(image_path: str) -> str:
    """Read image from disk and return base64 string for the API payload."""
    with open(image_path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def call_model_with_retry(
    image_path: str,
    max_retries: int = MAX_RETRIES,
    base_delay: float = BASE_DELAY
) -> str:
    """
    POST image + prompt to the Ollama /api/chat endpoint.
    Retries with linear back-off on any error.
    Returns raw model text on success, raises Exception after all retries.
    """

    payload = {
        'model': MODEL_NAME,
        'messages': [
            {
                'role': 'user',
                'content': SYSTEM_PROMPT,
                'images': [encode_image_b64(image_path)]
            }
        ],
        'stream': False,
        'options': {'temperature': 0.2}
    }

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.post(
            OLLAMA_API_URL,
            json=payload,
            headers={'ngrok-skip-browser-warning': 'true'},
            timeout=120
            )
            resp.raise_for_status()
            return resp.json()['message']['content']

        except requests.exceptions.HTTPError as e:
            last_err = e
            delay = base_delay * attempt
            print(f'  HTTP error {resp.status_code}: {e}. Retry {attempt}/{max_retries} in {delay:.1f}s...')
            time.sleep(delay)

        except requests.exceptions.ConnectionError as e:
            last_err = e
            delay = base_delay * attempt
            print(f'  Connection error (is ngrok tunnel alive?). Retry {attempt}/{max_retries} in {delay:.1f}s...')
            time.sleep(delay)

        except Exception as e:
            last_err = e
            delay = base_delay * attempt
            print(f'  Unexpected error: {e}. Retry {attempt}/{max_retries} in {delay:.1f}s...')
            time.sleep(delay)

    raise Exception(f'Failed after {max_retries} attempts. Last error: {last_err}')

In [ ]:
from pathlib import Path

SUPPORTED_EXTS = {'.png', '.jpg', '.jpeg'}

def process_images_recursively(directory: str):
    all_images = [
        Path(root) / file
        for root, _, files in os.walk(directory)
        for file in files
        if Path(file).suffix.lower() in SUPPORTED_EXTS
    ]

    total      = len(all_images)
    skipped    = 0
    succeeded  = 0
    failed     = 0

    print(f'Found {total} images in {directory}\n')

    for idx, image_path in enumerate(all_images, 1):
        output_filename = image_path.stem + '.json'
        output_path     = Path(OUTPUT_DIR) / output_filename

        # Skip already-annotated images
        if output_path.exists():
            skipped += 1
            continue

        print(f'[{idx}/{total}] {image_path.name} ...', end=' ', flush=True)
        t0 = time.time()

        try:
            raw_text   = call_model_with_retry(str(image_path))
            sentence   = clean_to_sentence(raw_text)
            annotation = wrap_as_annotation(sentence)

            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(annotation, f, indent=4, ensure_ascii=False)

            elapsed = time.time() - t0
            print(f'({elapsed:.1f}s) → {sentence[:80]}')
            succeeded += 1

        except Exception as e:
            print(f'permanent failure — logged.')
            log_failure(str(image_path), str(e))
            failed += 1
            continue

    return total, skipped, succeeded, failed

print('Starting annotation...\n')
total, skipped, succeeded, failed = process_images_recursively(EXDARK_IMAGE_DIR)
print('\n' + '='*60)
print(f'Done.  Total: {total}  |  New: {succeeded}  |  Skipped: {skipped}  |  Failed: {failed}')
if failed:
    print(f'Check {FAILED_LOG_PATH} for failed images.')

Starting annotation...

Found 7363 images in /content/drive/MyDrive/ExDark/ExDark/

[7/7363] 2015_07215.jpg ... permanent failure — logged.
[8/7363] 2015_07176.JPG ... permanent failure — logged.
[11/7363] 2015_07148.JPG ...   HTTP error 400: 400 Client Error: Bad Request for url: https://going-seventeen-deceit.ngrok-free.dev/api/chat. Retry 1/5 in 2.0s...
  HTTP error 400: 400 Client Error: Bad Request for url: https://going-seventeen-deceit.ngrok-free.dev/api/chat. Retry 2/5 in 4.0s...
  HTTP error 400: 400 Client Error: Bad Request for url: https://going-seventeen-deceit.ngrok-free.dev/api/chat. Retry 3/5 in 6.0s...
  HTTP error 400: 400 Client Error: Bad Request for url: https://going-seventeen-deceit.ngrok-free.dev/api/chat. Retry 4/5 in 8.0s...
  HTTP error 400: 400 Client Error: Bad Request for url: https://going-seventeen-deceit.ngrok-free.dev/api/chat. Retry 5/5 in 10.0s...
permanent failure — logged.
[12/7363] 2015_07177.JPG ... permanent failure — logged.
[13/7363] 2015_0701